# Détecteur et Classifieur d'Objets Spatiaux & Débris
Detecteur de Déchets 

Ce notebook prépare le pipeline complet de données avec **Pandas** et **TensorFlow**.
Chaque image contient un **unique objet** (satellite ou débris). Le pipeline génère ainsi deux cibles pour chaque image :
- `class_output` : l'identifiant de la classe (parmi 11 classes)
- `bbox_output` : les 4 coordonnées normalisées `[ymin, xmin, ymax, xmax]`

In [8]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import tensorflow as tf
import yaml

# Configuration globale
IMG_SIZE = 224  # Taille standard adaptée aux backbones CNN (ex: MobileNetV2, ResNet)
BATCH_SIZE = 16

DATA_YAML_PATH = 'data/data.yaml'
with open(DATA_YAML_PATH, 'r', encoding='utf-8') as yaml_file:
  dataset_config = yaml.safe_load(yaml_file)

CLASS_NAMES = dataset_config['names']
CLASS_NAMES
NUM_CLASSES = len(CLASS_NAMES)
print(f"TensorFlow version: {tf.__version__}")
print(f"Nombre de classes: {NUM_CLASSES}")


ModuleNotFoundError: No module named 'yaml'

## 1. Préparation des DataFrames Pandas

Nous lisons les annotations YOLO (`[class_id, x_center, y_center, width, height]`) et les convertissons directement en `[ymin, xmin, ymax, xmax]`.
Chaque image contenant un unique objet, nous n'avons plus besoin de padding `MAX_BOXES`.


In [ ]:
def create_dataset_dataframe(image_dir):
    """
    Lit les images et leurs étiquettes YOLO pour créer un DataFrame Pandas tabulaire.
    """
    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
    records = []

    for img_path in image_paths:
        label_path = img_path.replace('images', 'labels').replace('.jpg', '.txt')
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                line = f.readline().strip()
                if line:
                    parts = line.split()
                    cls_id = int(parts[0])
                    x_c, y_c, w, h = map(float, parts[1:5])

                    # Conversion YOLO (x_center, y_center, w, h) -> (ymin, xmin, ymax, xmax)
                    ymin = max(0.0, y_c - (h / 2.0))
                    xmin = max(0.0, x_c - (w / 2.0))
                    ymax = min(1.0, y_c + (h / 2.0))
                    xmax = min(1.0, x_c + (w / 2.0))

                    records.append({
                        'image_path': img_path,
                        'class_id': cls_id,
                        'class_name': CLASS_NAMES[cls_id] if 0 <= cls_id < len(CLASS_NAMES) else 'unknown',
                        'ymin': ymin,
                        'xmin': xmin,
                        'ymax': ymax,
                        'xmax': xmax,
                    })

    return pd.DataFrame(records)

# Création des DataFrames
df_train = create_dataset_dataframe('data/train/images')
df_val   = create_dataset_dataframe('data/valid/images')
df_test  = create_dataset_dataframe('data/test/images')

print(f"Train : {len(df_train)} images")
print(f"Val   : {len(df_val)} images")
print(f"Test  : {len(df_test)} images")
df_train.head()


## 2. Exploration des données (EDA)
Vérifions la distribution des classes dans l'ensemble d'entraînement.


In [ ]:
print("Distribution des classes (Train) :")
print(df_train['class_name'].value_counts())


,image_path,label_path,has_label,num_objects,classes_present,boxes_with_class
0,data/valid/images/img030241_jpg.rf.4ffebca7252...,data/valid/labels/img030241_jpg.rf.4ffebca7252...,True,1,[double_start],"[[0.39101562500000003, 0.1765625, 0.823046875,..."
1,data/valid/images/img100649_jpg.rf.d279efa402b...,data/valid/labels/img100649_jpg.rf.d279efa402b...,True,1,[xmm_newton],"[[0.23632812500000003, 0.34335937499999997, 0...."
2,data/valid/images/img041129_jpg.rf.fcf4f8e7250...,data/valid/labels/img041129_jpg.rf.fcf4f8e7250...,True,1,[earth_observation_sat_1],"[[0.56796875, 0.458203125, 0.85703125, 0.86679..."
3,data/valid/images/img000628_jpg.rf.6dc9522a3a0...,data/valid/labels/img000628_jpg.rf.6dc9522a3a0...,True,1,[proba_2],"[[0.19960937499999998, 0.11367187499999998, 0...."
4,data/valid/images/img000558_jpg.rf.ebcbfd72e0c...,data/valid/labels/img000558_jpg.rf.ebcbfd72e0c...,True,1,[proba_2],"[[0.015625, 0.025390625, 0.609375, 0.808984375..."
...,...,...,...,...,...,...
234,data/valid/images/img060533_jpg.rf.bf57257028a...,data/valid/labels/img060533_jpg.rf.bf57257028a...,True,1,[proba_3_csc],"[[0.28046875000000004, 0.26796875, 0.66015625,..."
235,data/valid/images/img040451_jpg.rf.812cfc46f35...,data/valid/labels/img040451_jpg.rf.812cfc46f35...,True,1,[earth_observation_sat_1],"[[0.03828124999999999, 0.1375, 0.33515625, 0.3..."
236,data/valid/images/img022166_jpg.rf.1a934c419d8...,data/valid/labels/img022166_jpg.rf.1a934c419d8...,True,1,[debris],"[[0.521484375, 0.2859375, 0.662890625, 0.47343..."
237,data/valid/images/img050520_jpg.rf.875f221b209...,data/valid/labels/img050520_jpg.rf.875f221b209...,True,1,[lisa_pathfinder],"[[0.15156250000000002, 0.395703125, 0.484375, ..."


## 3. Pipeline TensorFlow (`tf.data`)

Nous transformons directement les DataFrames Pandas en `tf.data.Dataset` optimisés via `tf.data.Dataset.from_tensor_slices` :
- Chargement et décodage multithreadés (`AUTOTUNE`)
- Redimensionnement à `IMG_SIZE` et normalisation dans `[0, 1]`
- Sortie au format dictionnaire `{'class_output': ..., 'bbox_output': ...}` prêt pour un modèle multi-tâches (classification + localisation)


In [ ]:
def df_to_tf_dataset(df, img_size=IMG_SIZE, batch_size=BATCH_SIZE, is_training=True):
    image_paths = df['image_path'].values
    bboxes = df[['ymin', 'xmin', 'ymax', 'xmax']].values.astype(np.float32)
    class_ids = df['class_id'].values.astype(np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, bboxes, class_ids))

    def load_and_preprocess(img_path, bbox, cls_id):
        img_raw = tf.io.read_file(img_path)
        img = tf.image.decode_jpeg(img_raw, channels=3)
        img = tf.image.resize(img, [img_size, img_size])
        img = tf.cast(img, tf.float32) / 255.0
        return img, {
            'class_output': cls_id,
            'bbox_output': bbox,
        }

    if is_training:
        dataset = dataset.shuffle(buffer_size=min(len(df), 1000), reshuffle_each_iteration=True)

    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = df_to_tf_dataset(df_train, is_training=True)
val_ds   = df_to_tf_dataset(df_val, is_training=False)
test_ds  = df_to_tf_dataset(df_test, is_training=False)

print("Datasets TensorFlow construits et prêts pour l'entraînement !")


Tout est prêt pour l'entraînement !


## 4. Visualisation d'un lot pour vérification
Affichons un lot d'entraînement avec l'étiquette de classe et la boîte englobante correspondante.


In [ ]:
for images, targets in train_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()

    for i in range(min(8, len(images))):
        img = images[i].numpy()
        cls_id = int(targets['class_output'][i].numpy())
        ymin, xmin, ymax, xmax = targets['bbox_output'][i].numpy()

        h, w, _ = img.shape
        box_y = ymin * h
        box_x = xmin * w
        box_h = (ymax - ymin) * h
        box_w = (xmax - xmin) * w

        axes[i].imshow(img)
        rect = patches.Rectangle((box_x, box_y), box_w, box_h, linewidth=2, edgecolor='lime', facecolor='none')
        axes[i].add_patch(rect)
        axes[i].set_title(f"{CLASS_NAMES[cls_id]}", color='white', backgroundcolor='black', fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()
    break


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import rgb_to_hsv
from PIL import Image

images_dir = Path("data/test/images")
image_paths = sorted(
    path for path in images_dir.iterdir()
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
)[:20]

# Bornes reglables des trois composantes HSV : H (teinte), S (saturation), V (luminosite).
# Pour les modifier, conserver des valeurs comprises entre 0.0 et 1.0.
hue_range = (0.0, 1.0)
saturation_range = (0.0, 1.0)
value_range = (0.15, 1.0)

# Reinitialise les resultats pour ne pas reutiliser ceux d'une execution precedente.
hsv_images = []

# Traite chaque image individuellement.
for image_path in image_paths:
    # Charge l'image en RGB et normalise ses pixels entre 0 et 1.
    arr = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.float32) / 255.0

    # Convertit l'image RGB dans l'espace HSV.
    hsv = rgb_to_hsv(arr)

    # Construit un masque qui conserve uniquement les pixels dans les plages HSV choisies.
    mask = (
        (hsv[:, :, 0] >= hue_range[0]) & (hsv[:, :, 0] <= hue_range[1])
        & (hsv[:, :, 1] >= saturation_range[0]) & (hsv[:, :, 1] <= saturation_range[1])
        & (hsv[:, :, 2] >= value_range[0]) & (hsv[:, :, 2] <= value_range[1])
    )

    # Met a zero les pixels du fond qui ne respectent pas les plages HSV.
    hsv_filtered = hsv.copy()
    hsv_filtered[~mask] = 0.0
    hsv_images.append(hsv_filtered)

# Affiche les images par pages de 10.
images_per_page = 10
for page_start in range(0, len(image_paths), images_per_page):
    page_paths = image_paths[page_start:page_start + images_per_page]
    page_hsv = hsv_images[page_start:page_start + images_per_page]
    figure, axes = plt.subplots(len(page_paths), 2, figsize=(10, 3 * len(page_paths)))
    axes = np.atleast_2d(axes)

    # Affiche chaque paire RGB/HSV avec le nom du fichier.
    for row, (image_path, hsv) in enumerate(zip(page_paths, page_hsv)):
        arr = np.asarray(Image.open(image_path).convert("RGB"), dtype=np.float32) / 255.0
        axes[row, 0].imshow(arr)
        axes[row, 0].set_title(f"RGB - {image_path.name}")
        axes[row, 1].imshow(hsv)
        axes[row, 1].set_title("HSV filtre HSV")
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")

    # Ajuste la mise en page, affiche la page et libere la figure.
    figure.tight_layout()
    plt.show()
    plt.close(figure)

print(f"{len(hsv_images)} images filtrees avec le reglage HSV.")